# Feature Engineering

## AI-Powered Customer Retention Intelligence Platform

### Objective

The objective of this phase is to transform cleaned transactional data into meaningful business features that improve the performance of predictive machine learning models.

Feature engineering converts raw customer activity into measurable indicators of customer behavior, purchasing patterns, engagement, and retention risk.

In [16]:
import os
import sys
import pandas as pd

# Project Path
project_root = r"D:\AI-Powered-Customer-Retention-Intelligence-Platform"

os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.analysis.eda_loader import load_cleaned_data

datasets = load_cleaned_data()

print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.


In [17]:
for name, df in datasets.items():
    print(f"{name:<15} {df.shape}")

customers       (99441, 5)
orders          (99441, 8)
order_items     (112650, 7)
payments        (103886, 5)
products        (32951, 9)
reviews         (99224, 7)
sellers         (3095, 4)
geolocation     (738332, 5)
translation     (71, 2)


In [18]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 1 : Delivery Days
# ==========================================================

orders = datasets["orders"].copy()

# Convert Date Columns
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

# Create Feature
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

print("=" * 60)
print("DELIVERY DAYS FEATURE CREATED")
print("=" * 60)

print(orders[
    [
        "order_id",
        "delivery_days"
    ]
].head())

print("\nSummary Statistics")
print(orders["delivery_days"].describe())

DELIVERY DAYS FEATURE CREATED
                           order_id  delivery_days
0  e481f51cbdc54678b7cc49136f2d6af7            8.0
1  53cdb2fc8bc7dce0b6741e2150273451           13.0
2  47770eb9100c2d0c44946d9cf07ec65d            9.0
3  949d5b44dbf5de918fe9c16f97b45f8a           13.0
4  ad21c59c0840e6cb83a9ceb5573f8159            2.0

Summary Statistics
count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64


In [19]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 2 : Order Value
# ==========================================================

import pandas as pd

order_items = datasets["order_items"].copy()

# Calculate Total Order Value
order_value = (
    order_items.groupby("order_id")["price"]
    .sum()
    .reset_index()
)

order_value.rename(
    columns={"price": "order_value"},
    inplace=True
)

print("=" * 60)
print("ORDER VALUE FEATURE CREATED")
print("=" * 60)

print(order_value.head())

print("\nSummary Statistics")
print(order_value["order_value"].describe())

ORDER VALUE FEATURE CREATED
                           order_id  order_value
0  00010242fe8c5a6d1ba2dd792cb16214        58.90
1  00018f77f2f0320c557190d7a144bdd3       239.90
2  000229ec398224ef6ca0657da4fc703e       199.00
3  00024acbcdf0a6daa1e931b038114c75        12.99
4  00042b26cf59d7ce69dfabb4e55b4fd9       199.90

Summary Statistics
count    98666.000000
mean       137.754076
std        210.645145
min          0.850000
25%         45.900000
50%         86.900000
75%        149.900000
max      13440.000000
Name: order_value, dtype: float64


In [20]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 3 : Total Customer Spend
# ==========================================================

orders = datasets["orders"].copy()
payments = datasets["payments"].copy()

# Total Payment Per Order
payment_summary = (
    payments.groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
)

# Merge Orders with Payments
customer_spend = (
    orders.merge(payment_summary, on="order_id")
)

# Total Spend Per Customer
customer_spend = (
    customer_spend.groupby("customer_id")["payment_value"]
    .sum()
    .reset_index()
)

customer_spend.rename(
    columns={"payment_value": "total_customer_spend"},
    inplace=True
)

print("=" * 60)
print("TOTAL CUSTOMER SPEND FEATURE CREATED")
print("=" * 60)

print(customer_spend.head())

print("\nSummary Statistics")
print(customer_spend["total_customer_spend"].describe())

TOTAL CUSTOMER SPEND FEATURE CREATED
                        customer_id  total_customer_spend
0  00012a2ce6f8dcda20d059ce98491703                114.74
1  000161a058600d5901f007fab4c27140                 67.41
2  0001fd6190edaaf884bcaf3d49edf079                195.42
3  0002414f95344307404f0ace7a26f1d5                179.35
4  000379cdec625522490c315e70c7a9fb                107.01

Summary Statistics
count    99440.000000
mean       160.990267
std        221.951257
min          0.000000
25%         62.010000
50%        105.290000
75%        176.970000
max      13664.080000
Name: total_customer_spend, dtype: float64


In [21]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 4 : Total Orders
# ==========================================================

orders = datasets["orders"].copy()

customer_orders = (
    orders.groupby("customer_id")
    .size()
    .reset_index(name="total_orders")
)

print("=" * 60)
print("TOTAL ORDERS FEATURE CREATED")
print("=" * 60)

print(customer_orders.head())

print("\nSummary Statistics")
print(customer_orders["total_orders"].describe())

TOTAL ORDERS FEATURE CREATED
                        customer_id  total_orders
0  00012a2ce6f8dcda20d059ce98491703             1
1  000161a058600d5901f007fab4c27140             1
2  0001fd6190edaaf884bcaf3d49edf079             1
3  0002414f95344307404f0ace7a26f1d5             1
4  000379cdec625522490c315e70c7a9fb             1

Summary Statistics
count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: total_orders, dtype: float64


In [22]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 5 : Average Order Value
# ==========================================================

average_order_value = (
    customer_spend.merge(
        customer_orders,
        on="customer_id"
    )
)

average_order_value["average_order_value"] = (
    average_order_value["total_customer_spend"]
    / average_order_value["total_orders"]
)

print("=" * 60)
print("AVERAGE ORDER VALUE FEATURE CREATED")
print("=" * 60)

print(
    average_order_value[
        [
            "customer_id",
            "average_order_value"
        ]
    ].head()
)

print("\nSummary Statistics")
print(
    average_order_value["average_order_value"].describe()
)

AVERAGE ORDER VALUE FEATURE CREATED
                        customer_id  average_order_value
0  00012a2ce6f8dcda20d059ce98491703               114.74
1  000161a058600d5901f007fab4c27140                67.41
2  0001fd6190edaaf884bcaf3d49edf079               195.42
3  0002414f95344307404f0ace7a26f1d5               179.35
4  000379cdec625522490c315e70c7a9fb               107.01

Summary Statistics
count    99440.000000
mean       160.990267
std        221.951257
min          0.000000
25%         62.010000
50%        105.290000
75%        176.970000
max      13664.080000
Name: average_order_value, dtype: float64


In [23]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 6 : Average Review Score
# ==========================================================

orders = datasets["orders"].copy()
reviews = datasets["reviews"].copy()

# Merge Orders with Reviews
customer_reviews = orders.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

# Calculate Average Review Score Per Customer
average_review = (
    customer_reviews.groupby("customer_id")["review_score"]
    .mean()
    .reset_index()
)

average_review.rename(
    columns={"review_score": "average_review_score"},
    inplace=True
)

print("=" * 60)
print("AVERAGE REVIEW SCORE FEATURE CREATED")
print("=" * 60)

print(average_review.head())

print("\nSummary Statistics")
print(average_review["average_review_score"].describe())

AVERAGE REVIEW SCORE FEATURE CREATED
                        customer_id  average_review_score
0  00012a2ce6f8dcda20d059ce98491703                   1.0
1  000161a058600d5901f007fab4c27140                   4.0
2  0001fd6190edaaf884bcaf3d49edf079                   5.0
3  0002414f95344307404f0ace7a26f1d5                   5.0
4  000379cdec625522490c315e70c7a9fb                   4.0

Summary Statistics
count    98673.000000
mean         4.086793
std          1.346274
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: average_review_score, dtype: float64


In [24]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 7 : Preferred Payment Method
# ==========================================================

orders = datasets["orders"].copy()
payments = datasets["payments"].copy()

# Merge Orders with Payments
customer_payments = orders.merge(
    payments[["order_id", "payment_type"]],
    on="order_id",
    how="left"
)

# Most Frequent Payment Method Per Customer
payment_preference = (
    customer_payments.groupby("customer_id")["payment_type"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    .reset_index()
)

payment_preference.rename(
    columns={"payment_type": "preferred_payment_method"},
    inplace=True
)

print("=" * 60)
print("PREFERRED PAYMENT METHOD FEATURE CREATED")
print("=" * 60)

print(payment_preference.head())

PREFERRED PAYMENT METHOD FEATURE CREATED
                        customer_id preferred_payment_method
0  00012a2ce6f8dcda20d059ce98491703              credit_card
1  000161a058600d5901f007fab4c27140              credit_card
2  0001fd6190edaaf884bcaf3d49edf079              credit_card
3  0002414f95344307404f0ace7a26f1d5                   boleto
4  000379cdec625522490c315e70c7a9fb                   boleto


In [25]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 8 : Customer Purchase Frequency
# ==========================================================

orders = datasets["orders"].copy()

purchase_frequency = (
    orders.groupby("customer_id")
    .size()
    .reset_index(name="purchase_frequency")
)

print("=" * 60)
print("PURCHASE FREQUENCY FEATURE CREATED")
print("=" * 60)

print(purchase_frequency.head())

print("\nSummary Statistics")
print(purchase_frequency["purchase_frequency"].describe())

PURCHASE FREQUENCY FEATURE CREATED
                        customer_id  purchase_frequency
0  00012a2ce6f8dcda20d059ce98491703                   1
1  000161a058600d5901f007fab4c27140                   1
2  0001fd6190edaaf884bcaf3d49edf079                   1
3  0002414f95344307404f0ace7a26f1d5                   1
4  000379cdec625522490c315e70c7a9fb                   1

Summary Statistics
count    99441.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: purchase_frequency, dtype: float64


In [26]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 9 : RFM Features
# ==========================================================

import pandas as pd

orders = datasets["orders"].copy()
customers = datasets["customers"].copy()
payments = datasets["payments"].copy()

# Convert Purchase Date
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

# Merge Orders + Customers
orders = orders.merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

# Merge Payment Value
orders = orders.merge(
    payments.groupby("order_id")["payment_value"]
    .sum()
    .reset_index(),
    on="order_id",
    how="left"
)

# Reference Date
snapshot_date = (
    orders["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)

# Create RFM Table
rfm = (
    orders.groupby("customer_unique_id")
    .agg(
        Recency=(
            "order_purchase_timestamp",
            lambda x: (snapshot_date - x.max()).days
        ),
        Frequency=(
            "order_id",
            "count"
        ),
        Monetary=(
            "payment_value",
            "sum"
        )
    )
    .reset_index()
)

print("=" * 60)
print("RFM FEATURES CREATED")
print("=" * 60)

print(rfm.head())

print("\nSummary Statistics")
print(rfm.describe())

RFM FEATURES CREATED
                 customer_unique_id  Recency  Frequency  Monetary
0  0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19
2  0000f46a3911fa3c0805444483337064      586          1     86.22
3  0000f6ccb0745a6a4b88665a16c9f078      370          1     43.62
4  0004aac84e0df4da2b147fca70cf8255      337          1    196.89

Summary Statistics
            Recency     Frequency      Monetary
count  96096.000000  96096.000000  96096.000000
mean     288.735691      1.034809    166.592492
std      153.414676      0.214384    231.428332
min        1.000000      1.000000      0.000000
25%      164.000000      1.000000     63.120000
50%      269.000000      1.000000    108.000000
75%      398.000000      1.000000    183.530000
max      773.000000     17.000000  13664.080000


In [27]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 10 : Customer Lifetime Value
# ==========================================================

clv = rfm.copy()

clv["Customer_Lifetime_Value"] = (
    clv["Monetary"] * clv["Frequency"]
)

print("=" * 60)
print("CUSTOMER LIFETIME VALUE CREATED")
print("=" * 60)

print(
    clv[
        [
            "customer_unique_id",
            "Customer_Lifetime_Value"
        ]
    ].head()
)

print("\nSummary Statistics")
print(clv["Customer_Lifetime_Value"].describe())

CUSTOMER LIFETIME VALUE CREATED
                 customer_unique_id  Customer_Lifetime_Value
0  0000366f3b9a7992bf8c76cfdf3221e2                   141.90
1  0000b849f77a49e4a4ce2b2a4ca5be3f                    27.19
2  0000f46a3911fa3c0805444483337064                    86.22
3  0000f6ccb0745a6a4b88665a16c9f078                    43.62
4  0004aac84e0df4da2b147fca70cf8255                   196.89

Summary Statistics
count    96096.000000
mean       178.644356
std        302.511812
min          0.000000
25%         63.230000
50%        108.770000
75%        188.000000
max      28659.060000
Name: Customer_Lifetime_Value, dtype: float64


In [28]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 12 : Final Customer Feature Table
# ==========================================================

# Merge CLV and Risk Features
customer_features = (
    clv.merge(
        risk[["customer_unique_id", "Customer_Risk"]],
        on="customer_unique_id",
        how="left"
    )
)

print("=" * 60)
print("FINAL CUSTOMER FEATURE TABLE")
print("=" * 60)

print(customer_features.head())

print("\nShape")
print(customer_features.shape)

print("\nColumns")
print(customer_features.columns.tolist())

NameError: name 'risk' is not defined

In [31]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 12 : Final Customer Feature Table
# ==========================================================

# ----------------------------------------------------------
# 1. Customer-Level Review Features
# ----------------------------------------------------------

customer_reviews = (
    orders[
        [
            "order_id",
            "customer_id"
        ]
    ]
    .merge(
        customers[
            [
                "customer_id",
                "customer_unique_id"
            ]
        ],
        on="customer_id",
        how="left"
    )
    .merge(
        reviews[
            [
                "order_id",
                "review_score"
            ]
        ],
        on="order_id",
        how="left"
    )
)

review_features = (
    customer_reviews
    .groupby("customer_unique_id")["review_score"]
    .mean()
    .reset_index()
)

review_features.rename(
    columns={
        "review_score": "average_review_score"
    },
    inplace=True
)


# ----------------------------------------------------------
# 2. Customer Payment Features
# ----------------------------------------------------------

customer_payment_data = (
    orders[
        [
            "order_id",
            "customer_id"
        ]
    ]
    .merge(
        customers[
            [
                "customer_id",
                "customer_unique_id"
            ]
        ],
        on="customer_id",
        how="left"
    )
    .merge(
        payments[
            [
                "order_id",
                "payment_type"
            ]
        ],
        on="order_id",
        how="left"
    )
)

payment_features = (
    customer_payment_data
    .groupby("customer_unique_id")["payment_type"]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty
        else "unknown"
    )
    .reset_index()
)

payment_features.rename(
    columns={
        "payment_type": "preferred_payment_method"
    },
    inplace=True
)


# ----------------------------------------------------------
# 3. Average Order Value
# ----------------------------------------------------------

average_order_features = (
    rfm[
        [
            "customer_unique_id",
            "Monetary",
            "Frequency"
        ]
    ]
    .copy()
)

average_order_features["average_order_value"] = (
    average_order_features["Monetary"]
    / average_order_features["Frequency"]
)


# ----------------------------------------------------------
# 4. Customer Lifetime Value
# ----------------------------------------------------------

average_order_features["customer_lifetime_value"] = (
    average_order_features["Monetary"]
)


# ----------------------------------------------------------
# 5. Create Customer Risk
# ----------------------------------------------------------

customer_risk = rfm[
    [
        "customer_unique_id",
        "Recency",
        "Frequency"
    ]
].copy()

customer_risk["Customer_Risk"] = "Medium"

customer_risk.loc[
    customer_risk["Recency"] > 180,
    "Customer_Risk"
] = "High"

customer_risk.loc[
    (customer_risk["Recency"] <= 60) &
    (customer_risk["Frequency"] >= 2),
    "Customer_Risk"
] = "Low"


# ----------------------------------------------------------
# 6. Final Customer Feature Table
# ----------------------------------------------------------

customer_features = (
    rfm[
        [
            "customer_unique_id",
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
    .merge(
        review_features,
        on="customer_unique_id",
        how="left"
    )
    .merge(
        payment_features,
        on="customer_unique_id",
        how="left"
    )
    .merge(
        average_order_features[
            [
                "customer_unique_id",
                "average_order_value",
                "customer_lifetime_value"
            ]
        ],
        on="customer_unique_id",
        how="left"
    )
    .merge(
        customer_risk[
            [
                "customer_unique_id",
                "Customer_Risk"
            ]
        ],
        on="customer_unique_id",
        how="left"
    )
)


# ----------------------------------------------------------
# 7. Validation
# ----------------------------------------------------------

print("=" * 70)
print("FINAL CUSTOMER FEATURE TABLE")
print("=" * 70)

print("\nShape:")
print(customer_features.shape)

print("\nColumns:")
print(customer_features.columns.tolist())

print("\nFirst 5 Records:")
print(customer_features.head())

print("\nMissing Values:")
print(customer_features.isnull().sum())

print("\nDuplicate Customers:")
print(
    customer_features[
        "customer_unique_id"
    ].duplicated().sum()
)

FINAL CUSTOMER FEATURE TABLE

Shape:
(96096, 9)

Columns:
['customer_unique_id', 'Recency', 'Frequency', 'Monetary', 'average_review_score', 'preferred_payment_method', 'average_order_value', 'customer_lifetime_value', 'Customer_Risk']

First 5 Records:
                 customer_unique_id  Recency  Frequency  Monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2      161          1    141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164          1     27.19   
2  0000f46a3911fa3c0805444483337064      586          1     86.22   
3  0000f6ccb0745a6a4b88665a16c9f078      370          1     43.62   
4  0004aac84e0df4da2b147fca70cf8255      337          1    196.89   

   average_review_score preferred_payment_method  average_order_value  \
0                   5.0              credit_card               141.90   
1                   4.0              credit_card                27.19   
2                   3.0              credit_card                86.22   
3                   4.0              cr

In [32]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 13 : Churn Target Variable
# ==========================================================

# Create a copy of the final customer feature table
model_data = customer_features.copy()

# ----------------------------------------------------------
# Define Churn Proxy
# ----------------------------------------------------------

CHURN_THRESHOLD_DAYS = 180

model_data["churn_label"] = (
    model_data["Recency"] > CHURN_THRESHOLD_DAYS
).astype(int)

# ----------------------------------------------------------
# Display Results
# ----------------------------------------------------------

print("=" * 70)
print("CHURN TARGET VARIABLE CREATED")
print("=" * 70)

print("\nChurn Label Distribution:")
print(
    model_data["churn_label"]
    .value_counts()
)

print("\nChurn Percentage:")
print(
    model_data["churn_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nSample Records:")
print(
    model_data[
        [
            "customer_unique_id",
            "Recency",
            "Customer_Risk",
            "churn_label"
        ]
    ].head(10)
)

CHURN TARGET VARIABLE CREATED

Churn Label Distribution:
churn_label
1    68352
0    27744
Name: count, dtype: int64

Churn Percentage:
churn_label
1    71.13
0    28.87
Name: proportion, dtype: float64

Sample Records:
                 customer_unique_id  Recency Customer_Risk  churn_label
0  0000366f3b9a7992bf8c76cfdf3221e2      161        Medium            0
1  0000b849f77a49e4a4ce2b2a4ca5be3f      164        Medium            0
2  0000f46a3911fa3c0805444483337064      586          High            1
3  0000f6ccb0745a6a4b88665a16c9f078      370          High            1
4  0004aac84e0df4da2b147fca70cf8255      337          High            1
5  0004bd2a26a76fe21f786e4fbd80607f      195          High            1
6  00050ab1314c0e55a6ca13cf7181fecf      181          High            1
7  00053a61a98854899e70ed204dd4bafe      232          High            1
8  0005e1862207bf6ccc02e4228effd9a0      592          High            1
9  0005ef4cd20d2893f0d9fbd94d3c0d97      220          High  

In [33]:
# ==========================================================
# FEATURE ENGINEERING
# Feature 14 : Target Variable Validation
# ==========================================================

print("=" * 70)
print("TARGET VARIABLE VALIDATION")
print("=" * 70)

# ----------------------------------------------------------
# 1. Check target values
# ----------------------------------------------------------

print("\nUnique Churn Labels:")
print(model_data["churn_label"].unique())


# ----------------------------------------------------------
# 2. Check class distribution
# ----------------------------------------------------------

print("\nChurn Label Counts:")

churn_counts = (
    model_data["churn_label"]
    .value_counts()
    .sort_index()
)

print(churn_counts)


# ----------------------------------------------------------
# 3. Check class percentages
# ----------------------------------------------------------

print("\nChurn Label Percentages:")

churn_percentages = (
    model_data["churn_label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print(churn_percentages)


# ----------------------------------------------------------
# 4. Check missing target values
# ----------------------------------------------------------

print("\nMissing Churn Labels:")

print(
    model_data["churn_label"].isnull().sum()
)


# ----------------------------------------------------------
# 5. Check target consistency with Recency
# ----------------------------------------------------------

print("\nTarget Consistency Check:")

expected_labels = (
    model_data["Recency"] > CHURN_THRESHOLD_DAYS
).astype(int)

print(
    "Mismatched Records:",
    (model_data["churn_label"] != expected_labels).sum()
)


# ----------------------------------------------------------
# 6. Overall validation
# ----------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET VALIDATION COMPLETED")
print("=" * 70)

TARGET VARIABLE VALIDATION

Unique Churn Labels:
[0 1]

Churn Label Counts:
churn_label
0    27744
1    68352
Name: count, dtype: int64

Churn Label Percentages:
churn_label
0    28.87
1    71.13
Name: proportion, dtype: float64

Missing Churn Labels:
0

Target Consistency Check:
Mismatched Records: 0

TARGET VALIDATION COMPLETED


In [34]:
# ==========================================================
# SAVE FINAL MODEL DATASET
# ==========================================================

model_data.to_csv(
    "data/processed/customer_retention_model_data.csv",
    index=False
)

print("=" * 70)
print("FINAL MODEL DATASET SAVED")
print("=" * 70)

print(
    "File:",
    "data/processed/customer_retention_model_data.csv"
)

print(
    "Shape:",
    model_data.shape
)

FINAL MODEL DATASET SAVED
File: data/processed/customer_retention_model_data.csv
Shape: (96096, 10)


In [35]:
import pandas as pd

model_data = pd.read_csv(
    "data/processed/customer_retention_model_data.csv"
)

print("=" * 60)
print("FINAL CUSTOMER RETENTION MODEL DATA")
print("=" * 60)

print("\nShape:")
print(model_data.shape)

print("\nColumns:")
print(model_data.columns.tolist())

print("\nMissing Values:")
print(model_data.isnull().sum())

print("\nDuplicate Rows:")
print(model_data.duplicated().sum())

FINAL CUSTOMER RETENTION MODEL DATA

Shape:
(96096, 10)

Columns:
['customer_unique_id', 'Recency', 'Frequency', 'Monetary', 'average_review_score', 'preferred_payment_method', 'average_order_value', 'customer_lifetime_value', 'Customer_Risk', 'churn_label']

Missing Values:
customer_unique_id            0
Recency                       0
Frequency                     0
Monetary                      0
average_review_score        716
preferred_payment_method      0
average_order_value           0
customer_lifetime_value       0
Customer_Risk                 0
churn_label                   0
dtype: int64

Duplicate Rows:
0
